# 🎵 MusicScope™ YouTube Dashboard

**Production-quality analytics for A&R decision-making**

---

## 📋 Executive Summary

This dashboard supports **6 artists** whose videos we're actively promoting for viral breakout.

**Key Question:** Should we lower our initial breakout velocity threshold from **75 → 55**?

**Data Source:** YouTube Data API v3 only (no Analytics API, no CTR/impressions, no revenue)

---

## 🎯 Dashboard Sections

1. **Artist & Audience Intelligence** — Roster overview, engagement patterns, current performance
2. **Breakout Threshold Experiment** — Compare outcomes at threshold 55 vs 75
3. **Budget Reallocation Strategies** — Three alternative approaches with pros/cons
4. **Fun FYI** — Additional insights (celebratory, non-blocking)

---

In [ ]:
# Set up Python path to include project root
import sys
from pathlib import Path

nb_cwd = Path.cwd()
project_root = nb_cwd.parent if nb_cwd.name == 'notebooks' else nb_cwd
sys.path.insert(0, str(project_root))

print(f"Project root added to path: {project_root}")
print(f"Current working directory: {nb_cwd}")

# Helper functions for clean charts
from textwrap import wrap

def clean_title(ax, title_text, max_width=50):
    """Set a clean, wrapped title that won't overlap"""
    wrapped = "\n".join(wrap(title_text, max_width))
    ax.set_title(wrapped, fontsize=11, fontweight='bold', pad=15)

def add_value_labels(ax, format_str="{:.0f}"):
    """Add value labels to bar charts"""
    for patch in ax.patches:
        if patch.get_width() > patch.get_height():  # horizontal bars
            value = patch.get_width()
            x = patch.get_width() + 0.01 * ax.get_xlim()[1]
            y = patch.get_y() + patch.get_height()/2
        else:  # vertical bars
            value = patch.get_height()
            x = patch.get_x() + patch.get_width()/2
            y = patch.get_height() + 0.01 * ax.get_ylim()[1]
        
        ax.annotate(format_str.format(value), (x, y), 
                   ha='left' if patch.get_width() > patch.get_height() else 'center',
                   va='center' if patch.get_width() > patch.get_height() else 'bottom',
                   fontsize=9, color='#333333')

# Color palette for consistency
CHART_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
# ═══════════════════════════════════════════════════════════════════════════
# CSS FIX FOR GREY TEXT ISSUE
# ═══════════════════════════════════════════════════════════════════════════

from IPython.display import HTML, display

# Fix grey text in all HTML displays
display(HTML("""
<style>
/* Force black text in all rendered content */
.jp-RenderedMarkdown, .jp-RenderedMarkdown * {
    color: #000000 !important;
}
.jp-RenderedHTMLCommon, .jp-RenderedHTMLCommon * {
    color: #000000 !important;
}
.jp-OutputArea-output {
    color: #000000 !important;
}
/* Specific fixes for custom HTML */
div[style*="background:#f9f9f9"] * {
    color: #000000 !important;
}
div[style*="background:#fff3e6"] * {
    color: #000000 !important;
}
div[style*="background:#e6fff9"] * {
    color: #000000 !important;
}
</style>
"""))

print("✅ CSS fix applied - text should now be black instead of grey")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG & HUMAN-REVIEW GATES
# ═══════════════════════════════════════════════════════════════════════════

# --- Artist Roster (HUMAN-REVIEW) ---
ARTISTS_OVERRIDE = [
    # Optional roster override — edit or leave empty to infer from data.
    # "hicorook", "Flyana Boss", "Raiche", "BiC Fizzle", "COBRAH", "re6ce"
]

# --- Approval Flags (HUMAN-REVIEW) ---
HUMAN_REVIEW_APPROVED = {
    "artists_override": False,   # set True after you review ARTISTS_OVERRIDE
    "budget_inputs":    False,   # set True after you set BUDGET amounts
}

# --- Thresholds ---
THRESHOLDS = {
    "legacy":       75,  # for comparison plots (historical)
    "pre_breakout": 55,  # BLUE highlight + invest-gradually signal
    "breakout":     60,  # legacy episode computation (kept for reference)
}


# --- Episode Detection Controls ---
EPISODE_THRESHOLD = THRESHOLDS.get("legacy", THRESHOLDS["breakout"])  # default to the stricter 75 viral bar
PREWARN_LOWER = THRESHOLDS["pre_breakout"]
PREWARN_MAX_DAYS = 30  # cap early-warning lookback so alerts stay tactical
SCMTV_BREAKOUT_FLOOR = max(THRESHOLDS["pre_breakout"], 60)  # require community energy, not just static KPIs

# --- Budget Parameters (HUMAN-REVIEW REQUIRED) ---
BUDGET = {
    # >>>>> HUMAN REVIEW REQUIRED — set your real amounts/percentages <<<<<
    "tier_55_pct":      None,  # e.g., 0.20 for 20% of pool
    "tier_75_pct":      None,  # e.g., 0.35
    "cap_per_artist":   None,  # e.g., 1500.0 (USD)
}

# --- Analysis Period ---
import pandas as pd
from datetime import datetime, timedelta

END_DATE = pd.Timestamp.now().normalize()
START_DATE = END_DATE - timedelta(days=90)  # 90-day analysis window

# --- Human-Review Badge (with forced black text) ---
def _hr_badge():
    msgs = []
    if ARTISTS_OVERRIDE and not HUMAN_REVIEW_APPROVED["artists_override"]:
        msgs.append("Artist roster override present but not approved.")
    if any(BUDGET[k] in (None, 0) for k in ["tier_55_pct","tier_75_pct","cap_per_artist"]) \
       and not HUMAN_REVIEW_APPROVED["budget_inputs"]:
        msgs.append("Budget inputs not approved or unset.")
    if msgs:
        from IPython.display import HTML, display
        html = "<br>".join(f"<span style='color:#000000 !important;'>• {m}</span>" for m in msgs)
        display(HTML(f'''
        <style>
        .warning-box, .warning-box * {{
            color: #000000 !important;
        }}
        </style>
        <div class="warning-box" style="padding:14px;border:3px solid #d95f02;border-radius:12px;background:#fff3e6;color:#000000 !important;">
          <b style="color:#000000 !important;">⚠️  HUMAN-REVIEW REQUIRED</b><br>{html}
        </div>
        '''))
    else:
        from IPython.display import HTML, display
        display(HTML(f'''
        <style>
        .success-box, .success-box * {{
            color: #000000 !important;
        }}
        </style>
        <div class="success-box" style="padding:14px;border:2px solid #1b9e77;border-radius:12px;background:#e6fff9;color:#000000 !important;">
          <b style="color:#000000 !important;">✅ All human-review gates approved</b>
        </div>
        '''))

_hr_badge()

print(f"\n📅 Analysis Period: {START_DATE.date()} → {END_DATE.date()}")
print(f"🎯 Thresholds: Pre-breakout={PREWARN_LOWER}, Breakout episodes={EPISODE_THRESHOLD}, Historical baseline={THRESHOLDS['breakout']}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — Imports & Global Style (Matplotlib-only, WCAG-minded)
# ═══════════════════════════════════════════════════════════════════════════
import os, sys, math, json, re, html
from typing import Optional
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.ticker import FuncFormatter

from src.youtubeviz.data_discovery import discover_data

# Accessible palette (no red/green)
BLUE  = "#1f77b4"
ORNG  = "#ff7f0e"
PURP  = "#7b3294"
GREY  = "#B0B0B0"
TXT   = "#222"
GRID  = "#e8e8e8"
BG    = "white"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.color": GRID,
    "grid.alpha": 0.3,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "font.size": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.autolayout": False,
    "legend.frameon": False,
})

print("✅ Imports loaded, style configured (Matplotlib-only)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Visualization Helpers (direct labels + date-year guard)
# ═══════════════════════════════════════════════════════════════════════════
from textwrap import shorten

def clean_title(ax, text: str):
    ax.set_title(shorten(text, width=140, placeholder="…"), fontweight="bold", color=TXT)

def add_value_labels(ax, fmt="{:,.0f}", padding=4, orientation="vertical"):
    """Label bar values directly (no legend scan). orientation: 'vertical'|'horizontal'"""
    for c in ax.containers:
        for bar in c:
            if orientation == "vertical":
                val = bar.get_height()
                if np.isnan(val):
                    continue
                ax.text(bar.get_x() + bar.get_width()/2, val, fmt.format(val),
                        ha="center", va="bottom", fontsize=10, color=TXT)
            else:
                val = bar.get_width()
                if np.isnan(val):
                    continue
                ax.text(val, bar.get_y() + bar.get_height()/2, fmt.format(val),
                        ha="left", va="center", fontsize=10, color=TXT)

def direct_line_label(ax, x, y, text, dx=6, dy=0):
    ax.annotate(text, xy=(x[-1], y[-1]), xytext=(dx,dy), textcoords="offset points",
                ha="left", va="center", fontsize=10, color=TXT, fontweight="bold")

def ensure_year_on_dates(ax):
    """Force year on dates (e.g., 'Jan 01 2025')."""
    ax.xaxis.set_major_formatter(DateFormatter("%b %d %Y"))

def robust_modified_z(series: pd.Series):
    """Modified Z (median/MAD); NaN-safe."""
    x = series.dropna()
    if x.empty:
        return pd.Series(index=series.index, dtype=float)
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0:
        return pd.Series(0.0, index=series.index)
    return 0.6745 * (series - med) / mad

print("✅ Helpers ready (direct labels, year guard, robust z)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VISUALIZATION HELPERS (Professional Standards)
# ═══════════════════════════════════════════════════════════════════════════

# --- Accessible Color Palette (4.5:1 contrast, no red/green combos) ---
PALETTE = [
    "#1b9e77",  # teal (primary signal)
    "#d95f02",  # orange (secondary signal)
    "#7570b3",  # purple (tertiary)
    "#e7298a",  # magenta (accent)
    "#66a61e",  # olive (context)
    "#e6ab02",  # gold (highlight)
]
GREY_1 = "#cccccc"  # light grey (context, non-data)
GREY_2 = "#888888"  # medium grey (neutral sentiment)
GREY_3 = "#444444"  # dark grey (text)
BLUE_BREAKOUT = "#3498db"  # BLUE for pre-breakout state (≥55)

def get_color(i: int) -> str:
    return PALETTE[i % len(PALETTE)]

def slide(figsize=(11,6), watermark: Optional[str]=None):
    """Create a professional slide-style figure."""
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_anchor("NW")
    if watermark:
        fig.text(0.5, 0.5, watermark, color=GREY_1, fontsize=60, ha="center",
                 va="center", alpha=0.35, rotation=30)
    return fig, ax

def action_title(ax: mpl.axes.Axes, finding: str, implication: str, action: str) -> None:
    """Set action-oriented title: Finding → Implication → Action."""
    ax.set_title(f"{finding} → {implication} → {action}", fontsize=13, fontweight="bold", pad=12)

def direct_line_labels(ax: mpl.axes.Axes, fontsize: int = 10):
    """Add direct labels to line chart (remove legend)."""
    lines = [ln for ln in ax.get_lines() if not ln.get_label().startswith("_")]
    for ln in lines:
        x, y = ln.get_xdata(), ln.get_ydata()
        if len(x)==0: continue
        ax.annotate(ln.get_label(), xy=(x[-1], y[-1]), xytext=(5,0), textcoords="offset points",
                    va="center", fontsize=fontsize, color=ln.get_color(), fontweight="bold")
    if ax.get_legend(): ax.get_legend().remove()

def label_bars(ax: mpl.axes.Axes, fmt="{:.0f}", fontsize=10):
    """Add value labels to bar chart."""
    for p in ax.patches:
        h = p.get_height()
        if h == 0: continue
        ax.text(p.get_x()+p.get_width()/2, p.get_y()+h, fmt.format(h),
                ha="center", va="bottom", fontsize=fontsize, color=GREY_3)

def iso8601_to_seconds(iso: str) -> int:
    """Parse ISO 8601 duration (PT1H2M3S) to seconds."""
    if not isinstance(iso, str): return 0
    h = m = s = 0
    mobj = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso)
    if mobj:
        h = int(mobj.group(1) or 0); m = int(mobj.group(2) or 0); s = int(mobj.group(3) or 0)
    return h*3600 + m*60 + s

print('✅ Visualization helpers loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA LOADING & VALIDATION
# ═══════════════════════════════════════════════════════════════════════════

# Discover and load data from YouTube API v3
chart_data = discover_data()

print('📈 Data Summary:')
for data_type, df in chart_data.items():
    print(f'   {data_type}: {len(df):,} rows, {len(df.columns)} columns')
    if 'artist_name' in df.columns:
        unique_artists = df['artist_name'].nunique()
        print(f'      → {unique_artists} unique artists')

# Extract core DataFrames
videos_df = chart_data.get('videos', pd.DataFrame())
comments_df = chart_data.get('comments', pd.DataFrame())
metrics_df = chart_data.get('metrics_timeseries', pd.DataFrame())

# Fix column name mismatch BEFORE validation
if 'comment_text' in comments_df.columns and 'text' not in comments_df.columns:
    comments_df['text'] = comments_df['comment_text']

# --- Data Contract Validation ---
REQUIRED_VIDEO_COLS = ['video_id', 'title', 'artist_name', 'published_at', 'view_count', 'like_count', 'comment_count']
REQUIRED_COMMENT_COLS = ['video_id', 'published_at', 'text']

missing_video_cols = set(REQUIRED_VIDEO_COLS) - set(videos_df.columns)
missing_comment_cols = set(REQUIRED_COMMENT_COLS) - set(comments_df.columns)

if missing_video_cols:
    raise ValueError(f"❌ Missing required video columns: {missing_video_cols}")
if missing_comment_cols:
    raise ValueError(f"❌ Missing required comment columns: {missing_comment_cols}")

# Ensure datetime types
videos_df['published_at'] = pd.to_datetime(videos_df['published_at'])
comments_df['published_at'] = pd.to_datetime(comments_df['published_at'])

# Determine artist roster
if ARTISTS_OVERRIDE:
    artists = ARTISTS_OVERRIDE
    print(f"\n🎵 Using override roster: {len(artists)} artists")
else:
    artists = sorted(videos_df['artist_name'].unique())
    print(f"\n🎵 Discovered roster: {len(artists)} artists")

for i, artist in enumerate(artists, 1):
    print(f"   {i}. {artist}")

ARTIST_COUNT = len(artists)

print(f"\n✅ Data loaded: {len(videos_df):,} videos, {len(comments_df):,} comments from {ARTIST_COUNT} artists")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA PREPARATION + SAFE COPIES
# ═══════════════════════════════════════════════════════════════════════════

# Create working copies
vids = videos_df.copy()
comments = comments_df.copy()

# Derived video features
vids["age_days"] = (END_DATE - vids["published_at"]).dt.days.clip(lower=1)
vids["views_per_day"] = (vids["view_count"] / vids["age_days"]).replace([np.inf, np.nan], 0.0)
vids["like_rate"] = (vids["like_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["comment_rate"] = (vids["comment_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["publish_week"] = vids["published_at"].dt.to_period("W").dt.to_timestamp()
vids["publish_month"] = vids["published_at"].dt.to_period("M").dt.to_timestamp()
vids["publish_hour"] = vids["published_at"].dt.hour
vids["publish_dow"] = vids["published_at"].dt.day_name()

# Comment features
if 'text' in comments.columns:
    comments["comment_length"] = comments["text"].str.len().fillna(0)

# ── Safe copies for downstream visuals ──────────────────────────────────────
vids_safe = vids.copy()
vids_safe["published_at"] = pd.to_datetime(vids_safe["published_at"], errors="coerce")
vids_safe = vids_safe.dropna(subset=["published_at"])
vids_safe["pub_date"] = vids_safe["published_at"].dt.floor("D")

comments_safe = None
if "comments_df" in globals():
    comments_safe = comments_df.copy()
elif "comments" in globals():
    comments_safe = comments.copy()

if comments_safe is not None:
    comments_safe["published_at"] = pd.to_datetime(comments_safe["published_at"], errors="coerce")
    comments_safe = comments_safe.dropna(subset=["published_at"])
    comments_safe["pub_date"] = comments_safe["published_at"].dt.floor("D")

print(f"✅ Data prepared: {len(vids):,} videos with derived features")
print(f"   Age range: {vids['age_days'].min():.0f} - {vids['age_days'].max():.0f} days")
print(f"   Views/day range: {vids['views_per_day'].min():.1f} - {vids['views_per_day'].max():.1f}")
print("✅ Metrics prepared (comments optional).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MOMENTUM SCORING (Cross-sectional daily normalization)
# ═══════════════════════════════════════════════════════════════════════════

# Daily comment counts per video
comments["_date"] = comments["published_at"].dt.floor("D")
c_daily = (comments.groupby(["video_id","_date"]).size()
                  .rename("comments_d").reset_index())

# Create daily panel for each video (clamped to analysis window)
rows = []
for _, vid in vids.iterrows():
    pub = vid["published_at"].floor("D")
    start = max(pub, START_DATE)
    if start > END_DATE:
        continue
    dates = pd.date_range(start, END_DATE, freq="D")
    for d in dates:
        rows.append({
            "video_id": vid["video_id"],
            "title": vid["title"],
            "artist_name": vid["artist_name"],
            "date": d,
            "views_per_day": vid["views_per_day"],
            "like_rate": vid["like_rate"],
        })

momentum_daily = pd.DataFrame(rows)

# Respect the analysis window explicitly in case upstream ETL widens the frame
if not momentum_daily.empty:
    momentum_daily = momentum_daily[(momentum_daily["date"] >= START_DATE) &
                                    (momentum_daily["date"] <= END_DATE)].copy()

# Merge comment counts
momentum_daily = momentum_daily.merge(c_daily, left_on=["video_id","date"], right_on=["video_id","_date"], how="left")
momentum_daily["comments_d"] = momentum_daily["comments_d"].fillna(0)

# Rolling comment velocity (14-day window)
momentum_daily = momentum_daily.sort_values(["video_id","date"])
momentum_daily["cmt_14d"] = momentum_daily.groupby("video_id")["comments_d"].transform(
    lambda x: x.rolling(14, min_periods=1).sum()
)
momentum_daily["comments_per_day_14d"] = (momentum_daily["cmt_14d"]/14.0).fillna(0)

# Robust per-day normalization (cross-sectional), then weighted momentum
def _score_component_daily(df, col):
    """Robust z-score → percentile → 0-100 scale."""
    med = df[col].median()
    mad = (df[col] - med).abs().median() + 1e-9
    z = (df[col] - med) / (1.4826*mad)
    return (z.rank(pct=True)*100).clip(0,100)

momentum_daily = momentum_daily.groupby("date", group_keys=False).apply(
    lambda d: d.assign(
        s_views=_score_component_daily(d, "views_per_day"),
        s_like=_score_component_daily(d, "like_rate"),
        s_cmtv=_score_component_daily(d, "comments_per_day_14d"),
    )
)

# Weighted momentum score (0-100)
momentum_daily["momentum_score"] = (0.45*momentum_daily["s_views"]
                                    +0.25*momentum_daily["s_like"]
                                    +0.30*momentum_daily["s_cmtv"]).round(1)

# State classification
tactical_breakout = momentum_daily["momentum_score"] >= EPISODE_THRESHOLD
momentum_daily["state"] = np.where(momentum_daily["momentum_score"]>=PREWARN_LOWER, "pre_breakout", "baseline")
momentum_daily["is_breakout"] = tactical_breakout.astype(int)

print(f"✅ Momentum calculated: {len(momentum_daily):,} daily observations")
print(f"   Score range: {momentum_daily['momentum_score'].min():.1f} - {momentum_daily['momentum_score'].max():.1f}")
print(f"   Pre-breakout days (≥{PREWARN_LOWER}): {(momentum_daily["momentum_score"]>=PREWARN_LOWER).sum():,}")
print(f"   Breakout days (≥{EPISODE_THRESHOLD}): {tactical_breakout.sum():,}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BREAKOUT EPISODE DETECTION
# ═══════════════════════════════════════════════════════════════════════════


def detect_episodes(df, video_id, threshold=None):
    """Detect contiguous breakout episodes for a music video within the analysis window.
    
    Args:
        df: Daily momentum DataFrame
        video_id: Unique video identifier
        threshold: Momentum score threshold for breakout (default: EPISODE_THRESHOLD)
    
    Returns:
        List of episode dicts with start_date, end_date, duration_days, pre_warning_hours
    """
    if threshold is None:
        threshold = EPISODE_THRESHOLD

    vid_data = df[df['video_id']==video_id].sort_values('date').copy()
    vid_data = vid_data[(vid_data['date'] >= START_DATE) & (vid_data['date'] <= END_DATE)]
    if vid_data.empty:
        return []

    # Require that momentum is both high and supported by community energy
    vid_data['is_breakout'] = (
        (vid_data['momentum_score'] >= threshold) &
        (vid_data['s_cmtv'] >= SCMTV_BREAKOUT_FLOOR)
    ).astype(int)

    # Find run starts and ends
    vid_data['run_start'] = (vid_data['is_breakout'] == 1) & (vid_data['is_breakout'].shift(1, fill_value=0) == 0)
    vid_data['run_end'] = (vid_data['is_breakout'] == 1) & (vid_data['is_breakout'].shift(-1, fill_value=0) == 0)

    episodes = []
    starts = vid_data[vid_data['run_start']]['date'].tolist()
    ends = vid_data[vid_data['run_end']]['date'].tolist()

    for start, end in zip(starts, ends):
        # Calculate pre-breakout warning hours (PREWARN_LOWER≤score<threshold immediately before episode)
        pre_window = vid_data[(vid_data['date'] < start) &
                              (vid_data['date'] >= start - pd.Timedelta(days=PREWARN_MAX_DAYS)) &
                              (vid_data['momentum_score'] >= PREWARN_LOWER) &
                              (vid_data['momentum_score'] < threshold)]

        # Find contiguous pre-breakout window immediately before start
        pre_window = pre_window.sort_values('date', ascending=False)
        contiguous_days = 0
        prev_date = start
        for _, row in pre_window.iterrows():
            if (prev_date - row['date']).days == 1:
                contiguous_days += 1
                prev_date = row['date']
            else:
                break
        pre_warning_hours = min(contiguous_days, PREWARN_MAX_DAYS) * 24

        # Clamp episode to the analysis window to prevent artificial inflation
        clamped_start = max(start, START_DATE)
        clamped_end = min(end, END_DATE)
        duration_days = max((clamped_end - clamped_start).days + 1, 0)

        episodes.append({
            'video_id': video_id,
            'start_date': clamped_start,
            'end_date': clamped_end,
            'duration_days': duration_days,
            'pre_warning_hours': pre_warning_hours,
        })

    return episodes

# Detect episodes for all music videos
all_episodes = []
for video_id in momentum_daily['video_id'].unique():
    episodes = detect_episodes(momentum_daily, video_id, threshold=EPISODE_THRESHOLD)
    all_episodes.extend(episodes)

episodes_df = pd.DataFrame(all_episodes)

if len(episodes_df) > 0:
    # Add video metadata (title, artist, content type)
    episodes_df = episodes_df.merge(
        vids[['video_id', 'title', 'artist_name']], 
        on='video_id', 
        how='left'
    )
    
    print(f"✅ Episodes detected: {len(episodes_df):,} breakout episodes across {episodes_df['video_id'].nunique()} music videos")
    print(f"   Artists with breakouts: {episodes_df['artist_name'].nunique()}")
    print(f"   Avg episode duration: {episodes_df['duration_days'].mean():.1f} days")
    print(f"   Avg pre-breakout warning: {episodes_df['pre_warning_hours'].mean():.1f} hours")
    print(f"   Total breakout days: {episodes_df['duration_days'].sum():,} days")
else:
    print(f"⚠️  No breakout episodes detected at threshold {EPISODE_THRESHOLD}")



In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# H1 — Artist-First Pledge (puts mission before metrics; white background)
# ───────────────────────────────────────────────────────────────────────────
from IPython.display import HTML, display

PRIMARY = "#1f77b4"  # safe blue
TEXT    = "#111"

display(HTML(f"""
<div style="
  background:#fff; color:{TEXT};
  border:2px solid {PRIMARY}; border-radius:16px;
  padding:18px 20px; box-shadow:0 6px 18px rgba(0,0,0,.06);
  font-family: ui-sans-serif, system-ui;">
  <h2 style="margin:0 0 8px 0;">🎤 Artist-First Pledge</h2>
  <p style="margin:0 0 10px 0;">
    These KPIs exist to <b>serve the art and the artist</b>. We use data to
    <i>protect focus, amplify momentum, and learn what helps fans connect</i>—never to reduce
    people to a number.
  </p>
  <ul style="margin:0 0 0 18px; line-height:1.5">
    <li><b>Respect:</b> Stories and context travel with the numbers.</li>
    <li><b>Clarity:</b> Direct labels & short annotations—no guesswork.</li>
    <li><b>Care:</b> We test gently at 55 (pre-breakout) and invest with intent at 60–75.</li>
  </ul>
</div>
"""))

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# H2 — Artist Spotlight Grid (human quotes + days at ≥55 pre-breakout)
# Requires: momentum_daily[date, video_id, momentum_score], vids[video_id, artist_name]
# Optional: comments_df[video_id, text, sentiment_score] or comments[text, ...]
# ───────────────────────────────────────────────────────────────────────────
import html
import pandas as pd
from IPython.display import HTML, display

THRESH_PRE = 55
PRIMARY = "#1f77b4"; ORANGE="#ff7f0e"; GREY="#9b9b9b"; TEXT="#111"


def _resolve_artist_column(df: pd.DataFrame) -> str:
    """Return the first artist-name column available after merges."""
    for col in ["artist_name", "artist_name_x", "artist_name_y", "artist", "artist_name_va"]:
        if col in df.columns:
            return col
    return ""

# Map videos → artists
if "vids" not in globals():
    raise RuntimeError("Missing vids DataFrame (needs video_id, artist_name).")
va = vids[["video_id","artist_name"]].drop_duplicates()

# Pre-breakout days per artist (unique days with any video ≥55)
if "momentum_daily" not in globals():
    raise RuntimeError("Missing momentum_daily DataFrame.")
md = momentum_daily.merge(va, on="video_id", how="left", suffixes=("","_va")).copy()
md["date"] = pd.to_datetime(md["date"]).dt.floor("D")

artist_col = _resolve_artist_column(md)
if not artist_col:
    raise RuntimeError("Could not determine artist column after merge.")

mask = md["momentum_score"] >= THRESH_PRE
per_day = md.loc[mask].groupby([artist_col, "date"]).size()
days_55 = per_day.groupby(level=0).size().astype(int).rename("pre_days")

# Full roster baseline (include artists with zero days)
roster = (pd.DataFrame({"artist_name": sorted(va["artist_name"].unique())})
            .set_index("artist_name"))
roster = roster.join(days_55, how="left").fillna(0).astype({"pre_days": int})

# Build quotes dictionary from best available comments source
quotes = {}
tmp = None
if "candidates" in globals() and isinstance(candidates, pd.DataFrame) and ("text" in candidates.columns):
    tmp = candidates.copy()
elif "comments_df" in globals() and isinstance(comments_df, pd.DataFrame) and ("text" in comments_df.columns):
    tmp = comments_df.copy()
elif "comments" in globals() and isinstance(comments, pd.DataFrame) and ("text" in comments.columns):
    tmp = comments.copy()

if tmp is not None:
    tmp = tmp.merge(va, on="video_id", how="left", suffixes=("","_va"))
    artist_col_tmp = _resolve_artist_column(tmp)
    if not artist_col_tmp:
        raise RuntimeError("Could not determine artist column for comments merge.")
    # Prefer positivity if available; otherwise longest text
    if "sentiment_score" in tmp.columns:
        tmp = tmp.sort_values([artist_col_tmp, "sentiment_score"], ascending=[True, False])
    else:
        tmp["len"] = tmp["text"].astype(str).str.len()
        tmp = tmp.sort_values([artist_col_tmp, "len"], ascending=[True, False])
    quotes = (tmp.dropna(subset=[artist_col_tmp, "text"])  
                .groupby(artist_col_tmp)["text"]
                .first()
                .to_dict())

# Render grid
cards = []
max_days = int(roster["pre_days"].max()) if len(roster) else 0
for _, row in roster.reset_index().sort_values("pre_days", ascending=False).iterrows():
    name = row["artist_name"]
    pre  = int(row["pre_days"])
    q    = quotes.get(name, None)
    qs   = html.escape(q[:160] + ("…" if q and len(q)>160 else "")) if q else "—"
    width = min(100, max(0, int((pre / max(max_days, 1)) * 100)))
    bar = f'<div style="height:8px;border-radius:6px;background:{GREY};">' \
          f'<div style="height:8px;width:{width}%;background:{PRIMARY};border-radius:6px;"></div></div>'
    cards.append(f"""
    <div style="border:2px solid {PRIMARY};border-radius:14px;padding:12px;background:#fff">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px">
        <div style="font-weight:700">{html.escape(name)}</div>
        <div style="color:{PRIMARY};font-weight:700">{pre} days ≥55</div>
      </div>
      {bar}
      <div style="margin-top:8px;color:#333"><em>“{qs}”</em></div>
    </div>
    """)

grid = f"""
<div style="font-family:ui-sans-serif,system-ui;color:{TEXT}">
  <h3 style="margin:0 0 10px 0;">🌟 Artist Spotlights — momentum & voices</h3>
  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px">{''.join(cards)}</div>
  <div style="margin-top:8px;color:#555;font-size:12px">
    Notes: “days ≥55” = pre-breakout days (momentum score ≥{THRESH_PRE}); quotes sampled from top supportive comments.
  </div>
</div>
"""
display(HTML(grid))

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# H3 — Quality helpers: contrast checker + robust outlier detector (MAD)
# ───────────────────────────────────────────────────────────────────────────
import re
import numpy as np
import pandas as pd

def _hex_to_rgb(h):
    h = h.strip()
    if not re.match(r"^#([0-9a-fA-F]{6})$", h):
        raise ValueError(f"Bad hex color: {h}")
    return tuple(int(h[i:i+2],16)/255.0 for i in (1,3,5))

def contrast_ratio(hex_fg, hex_bg="#FFFFFF"):
    # WCAG 2.1 relative luminance + contrast
    def _lum(c):
        def _c(v): 
            return v/12.92 if v <= 0.03928 else ((v+0.055)/1.055)**2.4
        r,g,b = (_c(v) for v in c)
        return 0.2126*r + 0.7152*g + 0.0722*b
    L1, L2 = _lum(_hex_to_rgb(hex_fg)), _lum(_hex_to_rgb(hex_bg))
    Lmax, Lmin = max(L1, L2), min(L1, L2)
    return (Lmax + 0.05) / (Lmin + 0.05)

def assert_contrast(hex_fg, hex_bg="#FFFFFF", min_ratio=4.5):
    r = contrast_ratio(hex_fg, hex_bg)
    if r < min_ratio:
        raise AssertionError(f"Contrast {r:.2f}:1 < {min_ratio}:1 (WCAG AA). Try darker text or lighter fill.")
    return r

def modified_z_outliers(series, thresh=3.5):
    """Return boolean mask for outliers via modified z-score (median/MAD)."""
    x = pd.Series(series).astype(float).dropna()
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0:
        return pd.Series(False, index=series.index)
    mz = 0.6745 * (series - med) / mad
    return mz.abs() > thresh

---

# 📊 Section 1: Artist & Audience Intelligence

Roster overview, engagement patterns, and current performance metrics.

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 1.1: Artist Roster Overview
# ═══════════════════════════════════════════════════════════════════════════
TEXT="#FFFFFF"

# Aggregate metrics by artist
artist_summary = vids.groupby('artist_name').agg({
    'video_id': 'count',
    'view_count': 'sum',
    'like_count': 'sum',
    'comment_count': 'sum',
    'views_per_day': 'mean',
    'like_rate': 'mean',
}).rename(columns={
    'video_id': 'videos',
    'view_count': 'total_views',
    'like_count': 'total_likes',
    'comment_count': 'total_comments',
    'views_per_day': 'avg_views_per_day',
    'like_rate': 'avg_like_rate',
}).round(2)

artist_summary = artist_summary.sort_values('total_views', ascending=False)

# Display as table
from IPython.display import display, HTML

html_table = artist_summary.to_html(classes='table table-striped', border=0)
display(HTML(f"""          
<div style="padding:20px;border:2px solid {PALETTE[0]};border-radius:12px;background:#f9f9f9,color:{TEXT};">
    <h3>🎵 Artist Roster Overview ({ARTIST_COUNT} Artists)</h3>
    {html_table}
</div>
"""))

print(f"\n✅ Roster summary: {ARTIST_COUNT} artists, {artist_summary['videos'].sum():.0f} total videos")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11 — Chart 1.2: Like-rate shows content quality → interactive scatter by artist
# Hover to see video titles; color = artist; size = view count
# ═══════════════════════════════════════════════════════════════════════════
try:
    import plotly.express as px
    import plotly.graph_objects as go  # noqa: F401 (reserved for extension)
    from plotly.subplots import make_subplots  # noqa: F401 (reserved for extension)
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("⚠️  Plotly not installed. Run: pip install plotly")

if PLOTLY_AVAILABLE:
    df = vids_safe.copy()

    if "like_rate" not in df.columns:
        df["like_rate"] = (df["like_count"] / df["view_count"].replace(0, np.nan)).fillna(0.0).clip(0, 1)

    if "title" not in df.columns:
        df["title"] = df.get("video_title", df.get("video_id", "Unknown"))

    required_cols = ["pub_date", "like_rate", "artist_name", "view_count", "title"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Missing required columns for Chart 1.2: {missing}")

    top_artists = (df.groupby("artist_name").size()
                     .sort_values(ascending=False).head(6).index.tolist())
    df_plot = df[df["artist_name"].isin(top_artists)].copy()

    df_plot["hover_text"] = (
        "<b>" + df_plot["title"].astype(str).str[:70] + "</b><br>" +
        "Artist: " + df_plot["artist_name"].astype(str) + "<br>" +
        "Like Rate: " + (df_plot["like_rate"] * 100).round(1).astype(str) + "%<br>" +
        "Views: " + df_plot["view_count"].apply(lambda x: f"{x:,}") + "<br>" +
        "Date: " + df_plot["pub_date"].dt.strftime("%b %d, %Y")
    )

    fig = px.scatter(
        df_plot,
        x="pub_date",
        y="like_rate",
        color="artist_name",
        size="view_count",
        hover_data={"hover_text": True, "pub_date": False, "like_rate": False,
                    "artist_name": False, "view_count": False},
        labels={"like_rate": "Like Rate", "pub_date": "Release Date"},
        title=(
            "Like-rate shows content quality → higher likes = audience values the work"
            "<br><sub>Hover for video titles | Size = views | Color = artist</sub>"
        ),
        color_discrete_sequence=px.colors.qualitative.Bold,
        size_max=24,
    )

    fig.update_yaxes(tickformat=".1%", title="Like Rate (%)")
    fig.update_xaxes(title="Release Date")

    fig.update_traces(hovertemplate="%{customdata[0]}<extra></extra>")

    fig.update_layout(
        height=600,
        font=dict(size=12, color=TXT),
        plot_bgcolor="white",
        paper_bgcolor="white",
        legend=dict(
            title="Artist",
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#cccccc",
            borderwidth=1
        ),
        hovermode="closest"
    )

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="#e8e8e8")
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="#e8e8e8")

    fig.show()
    print("✅ Chart 1.2: Interactive like-rate scatter (hover for video titles)")

else:
    print("📊 Falling back to matplotlib version...")
    df = vids_safe.copy()
    if "like_rate" not in df.columns:
        df["like_rate"] = (df["like_count"] / df["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)

    wk = (df.set_index("pub_date")
            .groupby("artist_name")["like_rate"]
            .resample("W").median()
            .reset_index())

    top_artists = (wk.groupby("artist_name")["like_rate"].count()
                     .sort_values(ascending=False).head(6).index.tolist())
    wk = wk[wk["artist_name"].isin(top_artists)]

    fig, ax = plt.subplots(figsize=(14,6))
    for artist, g in wk.groupby("artist_name"):
        g = g.sort_values("pub_date")
        y = (g["like_rate"]*100).values
        ax.plot(g["pub_date"], y, linewidth=2.2, label=artist, marker='o', markersize=4)

    ensure_year_on_dates(ax)
    ax.set_ylabel("Like Rate (%)", color=TXT, fontsize=11)
    ax.legend(title="Artist", loc="upper left", frameon=True, fancybox=False,
              edgecolor="#cccccc", fontsize=10)
    clean_title(ax, "Like-rate shows content quality → higher likes = audience values the work (weekly median)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    print("✅ Chart 1.2: Like-rate by artist (matplotlib fallback)")

In [ ]:
# ---- Choose how Chart 1.3 renders ----
# "exec"    = Beauty Pass (weekly, winsorized, clean)
# "both"    = Exec slide + Analyst appendix
# "analyst" = Analyst-only (daily, full rigor)
VIEW_MODE = "exec"
print(f"Chart 1.3 VIEW_MODE = {VIEW_MODE}")

### Chart 1.3 — What you’re looking at (Exec Beauty Pass)
- Left panel: Weekly median views/day. It calms noisy daily swings so leaders see the pacing trend at a glance.
- Right panel: Base‑100 index versus the weekly median. 100 = normal pace; above 100 = faster than normal; below 100 = slower than normal.
- Shading: Blue areas are above the baseline; light grey areas are below. The thin grey line marks the 100 (baseline) level.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 1.3 (Analyst, Daily Corrected): Robust daily + MAD outliers + TRUE base-100
# Wrapped so it only renders when VIEW_MODE requires it.
# ═══════════════════════════════════════════════════════════════════════════
if VIEW_MODE in ("both", "analyst"):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.dates import DateFormatter

    BLUE  = "#1f77b4"; ORANGE = "#ff7f0e"; RED = "#D95F02"; GREY = "#B0B0B0"; TXT = "#222"

    df = vids.copy()
    df["pub_date"] = df["published_at"].dt.floor("D")

    # Optional clamp to analysis window
    if "START_DATE" in globals() and "END_DATE" in globals():
        df = df[(df["pub_date"] >= START_DATE.floor("D")) & (df["pub_date"] <= END_DATE.floor("D"))]
    else:
        df = df[df["pub_date"] <= pd.Timestamp.today().normalize()]

    if "views_per_day" not in df.columns:
        _end = pd.Timestamp.today().normalize()
        df["age_days"] = (_end - df["published_at"]).dt.days.clip(lower=1)
        df["views_per_day"] = (df["view_count"] / df["age_days"]).replace([np.inf, np.nan], 0.0)

    if "like_rate" not in df.columns:
        df["like_rate"] = (df["like_count"] / df["view_count"].replace(0, np.nan)).fillna(0.0).clip(0, 1)

    if "comment_rate" not in df.columns:
        if "comment_count" in df.columns:
            if "age_days" not in df.columns:
                _end = pd.Timestamp.today().normalize()
                df["age_days"] = (_end - df["published_at"]).dt.days.clip(lower=1)
            df["comment_rate"] = (df["comment_count"] / df["age_days"]).replace([np.inf, np.nan], 0.0)
        else:
            df["comment_rate"] = np.nan

    daily_agg = (df.groupby("pub_date", as_index=True)
                   .agg(views_per_day=("views_per_day", "mean"),
                        like_rate=("like_rate", "mean"),
                        comment_rate=("comment_rate", "mean"))
                   .sort_index())

    W = 30
    daily_agg["views_per_day_30d"] = daily_agg["views_per_day"].rolling(W, min_periods=7).median()
    daily_agg["like_rate_30d"]     = daily_agg["like_rate"].rolling(W, min_periods=7).median()

    x   = daily_agg["views_per_day_30d"].dropna()
    med = x.median()
    mad = (x - med).abs().median()
    mz  = pd.Series(0.0, index=x.index) if mad == 0 else 0.6745 * (x - med) / mad
    outlier_dates = x[mz.abs() > 3.5].index.tolist()
    first_outlier = outlier_dates[np.argmax(x.loc[outlier_dates].values)] if len(outlier_dates) else None

    baseline = daily_agg["views_per_day"].rolling(W, min_periods=7).median()
    baseline = baseline.replace(0, np.nan).fillna(method="ffill").fillna(method="bfill")
    base100  = (daily_agg["views_per_day"] / baseline * 100).replace([np.inf, -np.inf], np.nan)

    if base100.notna().any():
        p99 = np.nanpercentile(base100.values, 99)
        base100 = base100.clip(upper=max(p99, 200))
    base100_smooth = base100.rolling(7, min_periods=3).median()

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10))

    ax1.set_facecolor("white")
    ax1.plot(daily_agg.index, daily_agg["views_per_day_30d"], color=BLUE, linewidth=2.5, zorder=3)
    ax1.fill_between(daily_agg.index, 0, daily_agg["views_per_day_30d"], alpha=0.20, color=BLUE, zorder=1)
    for d in outlier_dates:
        ax1.axvline(d, color=RED, linestyle="--", linewidth=1.2, alpha=0.8, zorder=2)
    title_tl = ("Robust trend shows viral spike in "
                f"{first_outlier.strftime('%b %Y')}" if first_outlier is not None
                else "Robust trend (30d median) shows no extreme outliers")
    ax1.set_title(f"{title_tl} → operationalize repeatable cues", fontweight="bold", color=TXT)
    ax1.set_ylabel("Avg Views/Day (30d rolling median)", fontsize=11, color=TXT)
    ax1.grid(True, alpha=0.3, zorder=0)
    ax1.xaxis.set_major_formatter(DateFormatter("%b %d %Y"))

    ax2.set_facecolor("white")
    ax2.plot(base100_smooth.index, base100_smooth, color=BLUE, linewidth=2.2)
    ax2.axhline(100, color=GREY, linewidth=1)
    ax2.fill_between(base100_smooth.index, 100, base100_smooth,
                     where=(base100_smooth >= 100), color=BLUE, alpha=0.18)
    ax2.fill_between(base100_smooth.index, base100_smooth, 100,
                     where=(base100_smooth < 100), color=GREY, alpha=0.18)
    ax2.set_title("Normalized to baseline: Base‑100 vs 30d median → 100 = normal", fontweight="bold", color=TXT)
    ax2.set_ylabel("Views/Day Index (Base = 100)", fontsize=11, color=TXT)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_formatter(DateFormatter("%b %d %Y"))

    ax3.set_facecolor("white")
    ax3.plot(daily_agg.index, daily_agg["like_rate_30d"] * 100, color=ORANGE, linewidth=2.5, zorder=3)
    if first_outlier is not None:
        ax3.axvline(first_outlier, color=RED, linestyle="--", linewidth=1.2, alpha=0.8, zorder=2)
    ax3.fill_between(daily_agg.index, 0, daily_agg["like_rate_30d"] * 100, alpha=0.20, color=ORANGE, zorder=1)
    ax3.set_ylabel("Avg Like Rate (%, 30d median)", fontsize=11, color=TXT)
    ax3.set_xlabel("Publish Date", fontsize=11, color=TXT)
    ax3.set_title("Like‑rate can dip during viral surges (casual viewers dilute engagement) — expected", fontweight="bold", color=TXT)
    ax3.grid(True, alpha=0.3)
    ax3.xaxis.set_major_formatter(DateFormatter("%b %d %Y"))

    ax4.set_facecolor("white")
    ax4.text(0.05, 0.95, "🔍 A&R INSIGHTS", fontsize=14, fontweight="bold", transform=ax4.transAxes, va="top")
    ax4.text(0.05, 0.85,
             f"• Viral spike around {first_outlier.strftime('%B %Y')}" if first_outlier is not None
             else "• No extreme spikes; compounding is steady",
             fontsize=11, transform=ax4.transAxes, va="top")
    ax4.text(0.05, 0.75, "• Base‑100: >100 above normal, <100 below", fontsize=11, transform=ax4.transAxes, va="top")
    ax4.text(0.05, 0.65, "• Replicate spike conditions (title/hook/thumbnail/feature)", fontsize=11, transform=ax4.transAxes, va="top")
    ax4.text(0.05, 0.55, "• Keep grey for context; highlight <10% of marks", fontsize=11, transform=ax4.transAxes, va="top")
    ax4.set_xlim(0, 1); ax4.set_ylim(0, 1); ax4.axis("off")

    plt.tight_layout()
    plt.show()

    print("✅ Chart 1.3 (Analyst Daily Corrected) rendered")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CHART 1.3 (Exec Beauty Pass): Weekly base-100 + release/context overlays
# Clarity goal: when we outperform baseline, who/what is driving it?
# ═══════════════════════════════════════════════════════════════════════════
if 'VIEW_MODE' not in globals():
    VIEW_MODE = 'exec'

if VIEW_MODE in ("exec", "both"):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib.dates import DateFormatter, MonthLocator
    import matplotlib.dates as mdates

    BLUE  = "#1f77b4"; ORANGE = "#ff7f0e"; MAGENTA = "#e7298a"; GREY = "#B0B0B0"; TXT = "#222"
    ALT_GREY = "#9b9b9b"

    base_df = vids_safe.copy() if 'vids_safe' in globals() else vids.copy()
    base_df = base_df.copy()

    if 'published_at' not in base_df.columns:
        raise RuntimeError("Missing 'published_at' in video data; run data prep cell first.")

    base_df['published_at'] = pd.to_datetime(base_df['published_at'], errors='coerce')
    base_df = base_df.dropna(subset=['published_at'])
    base_df['pub_week'] = base_df['published_at'].dt.to_period('W').dt.to_timestamp()

    if 'views_per_day' not in base_df.columns:
        ref_end = END_DATE if 'END_DATE' in globals() else pd.Timestamp.now().normalize()
        base_df['age_days'] = (ref_end - base_df['published_at']).dt.days.clip(lower=1)
        base_df['views_per_day'] = (base_df['view_count'] / base_df['age_days']).replace([np.inf, np.nan], 0.0)

    if 'START_DATE' in globals() and 'END_DATE' in globals():
        mask = (base_df['pub_week'] >= START_DATE.to_period('W').to_timestamp()) & \
               (base_df['pub_week'] <= END_DATE.to_period('W').to_timestamp())
        base_df = base_df.loc[mask]

    if base_df.empty:
        raise RuntimeError("No weekly data available for the selected window.")

    weekly_views = (base_df.groupby('pub_week')['views_per_day']
                              .mean()
                              .sort_index())
    releases_weekly = (base_df.groupby('pub_week')['video_id']
                                .nunique()
                                .reindex(weekly_views.index, fill_value=0))

    # Weekly baseline with outlier guard — cap at 90th percentile before rolling median
    valid_views = weekly_views.dropna()
    if not valid_views.empty:
        cap_value = np.nanpercentile(valid_views.values, 90)
        baseline_source = weekly_views.clip(upper=cap_value)
    else:
        baseline_source = weekly_views.copy()

    baseline = baseline_source.rolling(8, min_periods=4).median()
    base100 = (weekly_views / baseline * 100).replace([np.inf, -np.inf], np.nan)
    if base100.notna().any():
        cap = max(np.nanpercentile(base100.values, 99), 160)
        base100 = base100.clip(lower=0, upper=cap)
    base100_s = base100.rolling(3, min_periods=1).median()

    weekly_median_views = float(valid_views.median()) if not valid_views.empty else np.nan

    def _pick_content_column(df: pd.DataFrame) -> str:
        for candidate in [
            'content_type', 'content_category', 'topic_category', 'topic',
            'primary_tag', 'video_type', 'video_category'
        ]:
            if candidate in df.columns:
                return candidate
        df['content_proxy'] = 'Unlabeled'
        return 'content_proxy'

    content_col = _pick_content_column(base_df)
    base_df[content_col] = base_df[content_col].fillna('Unlabeled')

    content_week = (base_df.groupby(['pub_week', content_col])['views_per_day']
                              .sum()
                              .unstack(fill_value=0)
                              .reindex(weekly_views.index, fill_value=0))
    content_totals = content_week.sum().sort_values(ascending=False)
    keep_content = list(content_totals.head(4).index)
    other_content = content_week.drop(columns=keep_content, errors='ignore').sum(axis=1)
    content_plot = content_week[keep_content].copy() if keep_content else pd.DataFrame(index=weekly_views.index)
    if not other_content.empty and (other_content > 0).any():
        content_plot['Other'] = other_content
    content_plot = content_plot.fillna(0).rolling(4, min_periods=1).mean()

    artist_col = 'artist_name' if 'artist_name' in base_df.columns else None
    artist_plot = pd.DataFrame(index=weekly_views.index)
    if artist_col is not None:
        artist_week = (base_df.groupby(['pub_week', artist_col])['views_per_day']
                                .sum()
                                .unstack(fill_value=0)
                                .reindex(weekly_views.index, fill_value=0))
        top_artists = artist_week.sum().sort_values(ascending=False).head(5).index.tolist()
        other_artist = artist_week.drop(columns=top_artists, errors='ignore').sum(axis=1)
        artist_plot = artist_week[top_artists].copy()
        if (other_artist > 0).any():
            artist_plot['Other artists'] = other_artist
        artist_plot = artist_plot.fillna(0).rolling(4, min_periods=1).mean()

    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(16, 14), sharex=True, height_ratios=[2.2, 1.1, 1.1]
    )

    # Top panel — Base-100 performance + release cadence
    ax1.set_facecolor('white')
    ax1.axhline(100, color=GREY, linewidth=1, linestyle='--')
    ax1.fill_between(base100_s.index, 100, base100_s, where=(base100_s >= 100), color=BLUE, alpha=0.18)
    ax1.fill_between(base100_s.index, base100_s, 100, where=(base100_s < 100), color=ALT_GREY, alpha=0.16)
    ax1.plot(base100_s.index, base100_s, color=BLUE, linewidth=2.6, label='Weekly views/day index (base=100)')

    if len(base100_s.index) > 1:
        bar_width_days = np.diff(mdates.date2num(base100_s.index)).min()
    else:
        bar_width_days = 5
    release_series = releases_weekly.reindex(base100_s.index, fill_value=0)
    ax1_release = ax1.twinx()
    ax1_release.bar(
        base100_s.index,
        release_series.values,
        width=bar_width_days * 0.8,
        align='center',
        color=ORANGE,
        alpha=0.20,
        label='Videos released (weekly)'
    )
    ax1_release.set_ylabel('Videos released', color=ORANGE)
    ax1_release.tick_params(axis='y', colors=ORANGE)
    ax1_release.set_ylim(0, max(release_series.max() * 1.35, 1))

    non_null_base = base100_s.dropna()
    if not non_null_base.empty:
        crest_idx = non_null_base.idxmax()
        crest_value = float(non_null_base.loc[crest_idx])
        trough_idx = non_null_base.idxmin()
        trough_value = float(non_null_base.loc[trough_idx])
        ax1.annotate(
            f"Peak index {crest_value:.0f} (week of {crest_idx.strftime('%d %b %Y')})",
            xy=(crest_idx, crest_value), xytext=(15, 18), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.2), color=BLUE, fontsize=12,
        )
        ax1.annotate(
            f"Dip {trough_value:.0f} (week of {trough_idx.strftime('%d %b %Y')})",
            xy=(trough_idx, trough_value), xytext=(-60, -25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color=ALT_GREY, lw=1.2), color=ALT_GREY, fontsize=12,
        )

    title_suffix = (
        f"median weekly views/day ≈ {weekly_median_views:,.0f}"
        if not np.isnan(weekly_median_views)
        else "median weekly views/day unavailable"
    )
    ax1.set_title(
        f"Base-100 vs release cadence — baseline excludes >90th percentile spikes; {title_suffix}",
        fontweight='bold', color=TXT, fontsize=16
    )
    ax1.set_ylabel('Views/Day Index (Base = 100)', color=TXT, fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.xaxis.set_major_locator(MonthLocator(interval=1))
    ax1.xaxis.set_major_formatter(DateFormatter('%d %b %Y'))

    lines, labels = ax1.get_legend_handles_labels()
    bars, bar_labels = ax1_release.get_legend_handles_labels()
    if lines or bars:
        ax1.legend(lines + bars, labels + bar_labels, loc='upper left')

    # Middle panel — Mix of releases by content/category (who/what are we feeding the algorithm)
    ax2.set_facecolor('white')
    if not content_plot.empty:
        content_colors = [PALETTE[i % len(PALETTE)] for i in range(len(content_plot.columns))]
        if 'Other' in content_plot.columns:
            content_colors[content_plot.columns.get_loc('Other')] = ALT_GREY
        ax2.stackplot(
            content_plot.index,
            content_plot.T.values,
            labels=[str(c) for c in content_plot.columns],
            colors=content_colors,
            alpha=0.88
        )
        ax2.legend(loc='upper left', ncol=2)
        ax2.set_ylabel('Weekly views/day contribution', color=TXT, fontsize=12)
        ax2.set_title(
            'Music video still driving more views per video (4-week smoothed views/day)',
            fontweight='bold', color=TXT, fontsize=16
        )
        ax2.grid(True, alpha=0.25)
    else:
        ax2.text(0.5, 0.5, 'No content metadata available to segment mix.',
                 transform=ax2.transAxes, ha='center', va='center', color=TXT, fontsize=12)
        ax2.set_axis_off()

    # Bottom panel — Which artists are moving the index
    ax3.set_facecolor('white')
    if not artist_plot.empty:
        artist_colors = [PALETTE[i % len(PALETTE)] for i in range(len(artist_plot.columns))]
        if 'Other artists' in artist_plot.columns:
            artist_colors[artist_plot.columns.get_loc('Other artists')] = ALT_GREY
        ax3.stackplot(
            artist_plot.index,
            artist_plot.T.values,
            labels=[str(c) for c in artist_plot.columns],
            colors=artist_colors,
            alpha=0.88
        )
        ax3.legend(loc='upper left', ncol=2)
        ax3.set_ylabel('Weekly views/day contribution', color=TXT, fontsize=12)
        ax3.set_title(
            're6ce, Flyana Boss, BiC Fizzle leading weekly views/day (4-week smoothed)',
            fontweight='bold', color=TXT, fontsize=16
        )
        ax3.grid(True, alpha=0.25)
    else:
        ax3.text(0.5, 0.5, 'Artist attribution unavailable.',
                 transform=ax3.transAxes, ha='center', va='center', color=TXT, fontsize=12)
        ax3.set_axis_off()

    ax3.xaxis.set_major_locator(MonthLocator(interval=1))
    ax3.xaxis.set_major_formatter(DateFormatter('%d %b %Y'))

    plt.tight_layout()
    plt.show()

    print("✅ Chart 1.3 (Exec contextualized) rendered")

---

# 🎯 Section 2: Breakout Threshold Experiment Analysis

**Key Question:** Should we lower the initial breakout velocity threshold from **75 → 55**?

This section compares outcomes at both thresholds to enable data-driven decision-making.

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# THRESHOLD COMPARISON ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════

# Calculate metrics at both thresholds
threshold_55 = THRESHOLDS['pre_breakout']
threshold_75 = THRESHOLDS['legacy']

# Daily counts
qualifying_by_threshold_daily = momentum_daily.groupby('date').apply(
    lambda d: pd.Series({
        'videos_at_55': (d['momentum_score'] >= threshold_55).sum(),
        'videos_at_75': (d['momentum_score'] >= threshold_75).sum(),
        'artists_at_55': d[d['momentum_score'] >= threshold_55]['artist_name'].nunique(),
        'artists_at_75': d[d['momentum_score'] >= threshold_75]['artist_name'].nunique(),
    })
).reset_index()

# Overall summary
total_days_55 = (momentum_daily['momentum_score'] >= threshold_55).sum()
total_days_75 = (momentum_daily['momentum_score'] >= threshold_75).sum()
unique_videos_55 = momentum_daily[momentum_daily['momentum_score'] >= threshold_55]['video_id'].nunique()
unique_videos_75 = momentum_daily[momentum_daily['momentum_score'] >= threshold_75]['video_id'].nunique()
unique_artists_55 = momentum_daily[momentum_daily['momentum_score'] >= threshold_55]['artist_name'].nunique()
unique_artists_75 = momentum_daily[momentum_daily['momentum_score'] >= threshold_75]['artist_name'].nunique()

# Artist consistency (how many days each artist hits each threshold)
artist_consistency = momentum_daily.groupby('artist_name').apply(
    lambda d: pd.Series({
        'days_at_55': (d['momentum_score'] >= threshold_55).sum(),
        'days_at_75': (d['momentum_score'] >= threshold_75).sum(),
        'videos': d['video_id'].nunique(),
    })
).reset_index()

artist_consistency['consistency_ratio'] = (
    artist_consistency['days_at_75'] / artist_consistency['days_at_55'].replace(0, np.nan)
).fillna(0).round(2)

print("✅ Threshold analysis complete")
print(f"\n📊 Threshold Comparison Summary:")
print(f"   At threshold {threshold_55}:")
print(f"      → {total_days_55:,} qualifying video-days")
print(f"      → {unique_videos_55} unique videos")
print(f"      → {unique_artists_55} unique artists")
print(f"   At threshold {threshold_75}:")
print(f"      → {total_days_75:,} qualifying video-days")
print(f"      → {unique_videos_75} unique videos")
print(f"      → {unique_artists_75} unique artists")
print(f"   Incremental opportunity (55 vs 75):")
print(f"      → +{total_days_55 - total_days_75:,} video-days ({((total_days_55/total_days_75-1)*100 if total_days_75>0 else 0):.0f}% increase)")
print(f"      → +{unique_videos_55 - unique_videos_75} videos")
print(f"      → +{unique_artists_55 - unique_artists_75} artists")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 15 — Chart 2.1: Threshold comparison (55 vs 75) + Video-Day explainer
# ═══════════════════════════════════════════════════════════════════════════
from IPython.display import display, Markdown

THRESHOLDS = THRESHOLDS if "THRESHOLDS" in globals() else {"pre_breakout":55, "legacy":75, "breakout":60}
if "momentum_daily" not in globals():
    raise RuntimeError("momentum_daily missing — run momentum build first.")

display(Markdown("""
### 📊 **Video-Day (ELI-8)**
**Video-day** = a video meets the threshold for one calendar day.  
If 3 qualify Monday and 2 Tuesday → **5 video-days**.  
More video-days = more moments to activate.
"""))

def _daily_hits(d, t):
    return d.loc[d["momentum_score"]>=t].groupby("date")["video_id"].nunique().rename(f"videos_at_{int(t)}")

d55 = _daily_hits(momentum_daily, THRESHOLDS["pre_breakout"])
d75 = _daily_hits(momentum_daily, THRESHOLDS["legacy"])
q = pd.concat([d55, d75], axis=1).fillna(0).reset_index().sort_values("date")

# ✅ named columns (no positional iloc)
q["incremental_videos"] = q[f"videos_at_{int(THRESHOLDS['pre_breakout'])}"] - q[f"videos_at_{int(THRESHOLDS['legacy'])}"]
q["cumulative_incremental"] = q["incremental_videos"].cumsum()

ever55 = set(momentum_daily.loc[momentum_daily["momentum_score"]>=THRESHOLDS["pre_breakout"], "video_id"])
ever75 = set(momentum_daily.loc[momentum_daily["momentum_score"]>=THRESHOLDS["legacy"], "video_id"])
u55, u75 = len(ever55), len(ever75)

artists_55 = artists_75 = None
if "artist_name" in vids_safe.columns:
    m = vids_safe[["video_id","artist_name"]].drop_duplicates()
    artists_55 = m[m["video_id"].isin(ever55)]["artist_name"].nunique()
    artists_75 = m[m["video_id"].isin(ever75)]["artist_name"].nunique()

fig, (ax1, ax2) = plt.subplots(1,2, figsize=(16,7))

metrics, v55, v75 = ["Music Videos"], [u55], [u75]
if artists_55 is not None:
    metrics += ["Artists"]; v55 += [artists_55]; v75 += [artists_75]

x = np.arange(len(metrics)); w = 0.38
ax1.bar(x-w/2, v55, w, color=BLUE)
ax1.bar(x+w/2, v75, w, color=ORNG)
add_value_labels(ax1)
ax1.set_xticks(x); ax1.set_xticklabels(metrics)
ax1.set_ylabel("Count", color=TXT)
clean_title(ax1, f"Lowering to {THRESHOLDS['pre_breakout']} unlocks +{u55-u75:,} videos → more shots on goal")

ax2.plot(q["date"], q["cumulative_incremental"], linewidth=2.8, color=BLUE)
ax2.fill_between(q["date"], 0, q["cumulative_incremental"], color=BLUE, alpha=0.18)
ensure_year_on_dates(ax2)
ax2.set_ylabel("Cumulative Additional Video-Days", color=TXT)
clean_title(ax2, f"Threshold {THRESHOLDS['pre_breakout']} adds {int(q['incremental_videos'].sum()):,} video-days → align ops to activate")

plt.tight_layout(); plt.show()
print("✅ Chart 2.1 rendered")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 16 — KPI-22: Breakout duration (days) + Pre-warning (hours)
# • Breakout episode = contiguous days with score ≥ BRK (default 60)
# • Pre-warning window = contiguous days immediately before start with PRE≤score<BRK
# • Visual: top-10 artists (mean duration | mean pre-warning), direct-labeled barh
# • Cap handling: pre-warning capped at CAP_HOURS, hatched + annotated
# ═══════════════════════════════════════════════════════════════════════════
PRE, BRK, CAP_HOURS = 55, 60, 720  # 30 days cap

if "momentum_daily" not in globals():
    raise RuntimeError("momentum_daily missing.")

md = momentum_daily.copy()
md["date"] = pd.to_datetime(md["date"]).dt.floor("D")
md = md.sort_values(["video_id","date"])

# Map video -> artist for labeling
va = vids_safe[["video_id","artist_name"]].drop_duplicates() if "artist_name" in vids_safe.columns else None
def _artist_of(vid):
    if va is None:
        return "unknown"
    row = va.loc[va["video_id"] == vid]
    return row["artist_name"].iloc[0] if not row.empty else "unknown"

# Build fast lookup of scores per video/date
episodes = []
for vid, g in md.groupby("video_id", sort=False):
    g2 = g[["date","momentum_score"]].sort_values("date").set_index("date")
    dates = g2.index.tolist()
    in_ep = False
    start = None
    for i, d in enumerate(dates):
        s = float(g2.loc[d, "momentum_score"])
        if s >= BRK and not in_ep:
            in_ep = True
            start = d
        elif (s < BRK or i == len(dates)-1) and in_ep:
            end = dates[i] if (s >= BRK and i == len(dates)-1) else dates[i-1]
            pre_days = 0
            step = 1
            while True:
                prev_day = start - pd.Timedelta(days=step)
                if prev_day not in g2.index:
                    break
                prev_s = float(g2.loc[prev_day, "momentum_score"])
                if PRE <= prev_s < BRK:
                    pre_days += 1
                    step += 1
                else:
                    break
            episodes.append({
                "video_id": vid,
                "artist_name": _artist_of(vid),
                "start": start,
                "end": end,
                "duration_days": int((end - start).days + 1),
                "pre_warning_hours": int(min(pre_days * 24, CAP_HOURS)),
                "capped": pre_days * 24 >= CAP_HOURS,
            })
            in_ep = False

episodes_df = pd.DataFrame(episodes)
if episodes_df.empty:
    print("⚠️  KPI-22: no breakouts detected (score ≥ 60).")
else:
    agg = (
        episodes_df.groupby("artist_name")
        .agg(
            mean_duration_days=("duration_days", "mean"),
            mean_pre_warning_hours=("pre_warning_hours", "mean"),
            any_capped=("capped", "max"),
        )
        .reset_index()
    )

    top_dur = agg.nlargest(10, "mean_duration_days").sort_values("mean_duration_days")
    top_warn = agg.nlargest(10, "mean_pre_warning_hours").sort_values("mean_pre_warning_hours")

    overall_dur = agg["mean_duration_days"].mean()
    overall_warn = agg["mean_pre_warning_hours"].mean()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), sharex=False)

    # Left: duration
    y = np.arange(len(top_dur))
    bars_dur = ax1.barh(y, top_dur["mean_duration_days"], color=ORNG, alpha=0.9, height=0.6)
    ax1.set_yticks(y)
    ax1.set_yticklabels(top_dur["artist_name"], fontsize=10)
    ax1.set_xlabel("Avg Breakout Duration (days)", fontsize=11)
    ax1.set_title(
        f"KPI-22 (1/2): Breakouts last {overall_dur:.1f} days on average\n→ plan sustain tactics",
        fontweight="bold",
        fontsize=12,
        color=TXT,
        pad=12,
    )
    add_value_labels(ax1, fmt="{:.1f}", orientation="horizontal")
    ax1.grid(True, axis="x", alpha=0.3, color=GRID)

    # Right: pre-warning (cap-aware)
    y2 = np.arange(len(top_warn))
    bars_warn = ax2.barh(y2, top_warn["mean_pre_warning_hours"], color=PURP, alpha=0.9, height=0.6)
    cap_lookup = agg.set_index("artist_name")["any_capped"].to_dict()
    for idx, artist in enumerate(top_warn["artist_name"]):
        if cap_lookup.get(artist, False):
            bars_warn[idx].set_hatch("///")
    ax2.set_yticks(y2)
    ax2.set_yticklabels(top_warn["artist_name"], fontsize=10)
    ax2.set_xlabel("Avg Pre-Breakout Warning (hours)", fontsize=11)
    cap_note = "\n(hatched bars capped at ≥720h)" if any(cap_lookup.values()) else ""
    ax2.set_title(
        f"KPI-22 (2/2): Pre-breakout window ≈ {overall_warn:.0f}h on avg\n→ act inside window{cap_note}",
        fontweight="bold",
        fontsize=12,
        color=TXT,
        pad=12,
    )
    add_value_labels(ax2, fmt="{:.0f}", orientation="horizontal")
    ax2.grid(True, axis="x", alpha=0.3, color=GRID)

    plt.tight_layout()
    plt.show()
    print("✅ KPI-22 rendered (top-10 bars, contiguous logic, cap-aware)")

---

# 💰 Section 3: Budget Reallocation Strategy Comparison

Three alternative strategies for budget allocation based on momentum thresholds.

**⚠️  HUMAN-REVIEW REQUIRED:** Budget parameters must be set in the config cell before running this section.

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUDGET STRATEGY VALIDATION
# ═══════════════════════════════════════════════════════════════════════════

# Check if budget inputs are approved
if not HUMAN_REVIEW_APPROVED['budget_inputs']:
    from IPython.display import HTML, display
    display(HTML(f'''
    <div style="padding:20px;border:3px solid #d95f02;border-radius:12px;background:#000000">
        <h3>⚠️  HUMAN-REVIEW REQUIRED</h3>
        <p><b>Budget parameters are not set or approved.</b></p>
        <p>Please update the BUDGET dictionary in the config cell and set <code>HUMAN_REVIEW_APPROVED['budget_inputs'] = True</code></p>
        <p>This section will display placeholder analysis only.</p>
    </div>
    '''))
    BUDGET_APPROVED = False
else:
    BUDGET_APPROVED = True
    print("✅ Budget parameters approved")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUDGET STRATEGY CALCULATIONS
# ═══════════════════════════════════════════════════════════════════════════

# Ensure artist_consistency exists (build from momentum_daily if missing)
if 'artist_consistency' not in globals():
    if 'momentum_daily' in globals():
        # use configured thresholds if available
        pre_t = THRESHOLDS.get('pre_breakout', 55) if 'THRESHOLDS' in globals() else 55
        leg_t = THRESHOLDS.get('legacy', 75) if 'THRESHOLDS' in globals() else 75

        artist_consistency = momentum_daily.groupby('artist_name').apply(
            lambda d: pd.Series({
                'days_at_55': int((d['momentum_score'] >= pre_t).sum()) if 'momentum_score' in d.columns else 0,
                'days_at_75': int((d['momentum_score'] >= leg_t).sum()) if 'momentum_score' in d.columns else 0,
                'videos': int(d['video_id'].nunique()) if 'video_id' in d.columns else 0,
            })
        ).reset_index()
    else:
        raise RuntimeError("artist_consistency not found and momentum_daily is not available. Run momentum build cells first.")

# Strategy A: Tiered Escalation
# Gradual budget increase at 55, additional funding at 75
strategy_a = artist_consistency.copy()
# Ensure required columns exist
for c in ['days_at_55', 'days_at_75']:
    if c not in strategy_a.columns:
        strategy_a[c] = 0

if 'BUDGET_APPROVED' not in globals():
    BUDGET_APPROVED = False

if BUDGET_APPROVED and 'BUDGET' in globals():
    strategy_a['tier_55_allocation'] = strategy_a['days_at_55'] * float(BUDGET.get('tier_55_pct', 0))
    strategy_a['tier_75_allocation'] = strategy_a['days_at_75'] * float(BUDGET.get('tier_75_pct', 0))
    cap = BUDGET.get('cap_per_artist', None)
    if cap is not None:
        strategy_a['total_allocation'] = (strategy_a['tier_55_allocation'] + strategy_a['tier_75_allocation']).clip(upper=float(cap))
    else:
        strategy_a['total_allocation'] = strategy_a['tier_55_allocation'] + strategy_a['tier_75_allocation']
else:
    # placeholders when budget not approved
    strategy_a['tier_55_allocation'] = strategy_a['days_at_55'] * 0.20  # placeholder unit
    strategy_a['tier_75_allocation'] = strategy_a['days_at_75'] * 0.35  # placeholder unit
    strategy_a['total_allocation'] = strategy_a['tier_55_allocation'] + strategy_a['tier_75_allocation']

strategy_a = strategy_a.sort_values('total_allocation', ascending=False)

# Strategy B: Consistency Bonus
# Reward artists who consistently reach 55 AND routinely achieve 75
strategy_b = artist_consistency.copy()
for c in ['days_at_55', 'days_at_75']:
    if c not in strategy_b.columns:
        strategy_b[c] = 0

max_days_55 = max(strategy_b['days_at_55'].max(), 1)
max_days_75 = max(strategy_b['days_at_75'].max(), 1)
strategy_b['consistency_score'] = (
    0.6 * (strategy_b['days_at_55'] / max_days_55) +
    0.4 * (strategy_b['days_at_75'] / max_days_75)
)

if BUDGET_APPROVED and 'BUDGET' in globals() and BUDGET.get('cap_per_artist') is not None:
    total_pool = float(BUDGET.get('cap_per_artist')) * int(ARTIST_COUNT)
    denom = strategy_b['consistency_score'].sum() if strategy_b['consistency_score'].sum() > 0 else 1.0
    strategy_b['allocation'] = (strategy_b['consistency_score'] / denom) * total_pool
    strategy_b['allocation'] = strategy_b['allocation'].clip(upper=float(BUDGET.get('cap_per_artist')))
else:
    strategy_b['allocation'] = strategy_b['consistency_score'] * 1000  # placeholder

strategy_b = strategy_b.sort_values('allocation', ascending=False)

# Strategy C: Consistency-Based Allocation
# Rank by frequency of reaching 55, allocate proportionally
strategy_c = artist_consistency.copy()
if 'days_at_55' not in strategy_c.columns:
    strategy_c['days_at_55'] = 0

strategy_c['rank'] = strategy_c['days_at_55'].rank(ascending=False, method='dense')
# avoid division by zero
rank_sum = strategy_c['rank'].sum() if strategy_c['rank'].sum() > 0 else 1.0
strategy_c['rank_weight'] = (int(ARTIST_COUNT) + 1 - strategy_c['rank']) / rank_sum

if BUDGET_APPROVED and 'BUDGET' in globals() and BUDGET.get('cap_per_artist') is not None:
    total_pool = float(BUDGET.get('cap_per_artist')) * int(ARTIST_COUNT)
    strategy_c['allocation'] = (strategy_c['rank_weight'] * total_pool).clip(upper=float(BUDGET.get('cap_per_artist')))
else:
    strategy_c['allocation'] = strategy_c['rank_weight'] * 5000  # placeholder

strategy_c = strategy_c.sort_values('allocation', ascending=False)

print("✅ Budget strategies calculated")
if not BUDGET_APPROVED:
    print("⚠️  Using placeholder values (budget not approved)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# S1 — Budget Strategy Comparison (A/B/C)
# Builds strategy tables from momentum_daily + vids, then draws clean bar charts.
# Self-contained: safe defaults, direct labels, strategic greys, AA contrast mindset.
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Guards & inputs -------------------------------------------------------
if "momentum_daily" not in globals() or "vids" not in globals():
    raise RuntimeError("Need momentum_daily and vids in memory before running S1.")

THRESH_PRE = 55
THRESH_75  = 75
THRESH_EP  = 60

# Map video -> artist (fallback to 'unknown')
if "artist_name" in vids.columns:
    map_va = vids[["video_id","artist_name"]].drop_duplicates()
else:
    map_va = vids.assign(artist_name="unknown")[["video_id","artist_name"]].drop_duplicates()

# Ensure md has a single, non-null artist_name after merge
def _merge_artist(mdf, mapping):
    if 'artist_name' in mdf.columns:
        m = mdf.merge(mapping, on='video_id', how='left', suffixes=('', '_from_vids'))
        m['artist_name'] = m['artist_name'].fillna(m.get('artist_name_from_vids'))
        return m.drop(columns=[c for c in ['artist_name_from_vids'] if c in m.columns])
    return mdf.merge(mapping, on='video_id', how='left')

md = _merge_artist(momentum_daily.copy(), map_va)
md['date'] = pd.to_datetime(md['date']).dt.floor('D')

# Robust per-artist qualifying days (unique artist-date pairs)
md55 = md.loc[md['momentum_score'] >= THRESH_PRE, ['artist_name', 'date']].dropna().drop_duplicates()
md75 = md.loc[md['momentum_score'] >= THRESH_75,  ['artist_name', 'date']].dropna().drop_duplicates()
days_55 = md55.groupby('artist_name')['date'].nunique()
days_75 = md75.groupby('artist_name')['date'].nunique()

# Build roster from merged data (fallback to 'unknown')
tbl = pd.DataFrame({'artist_name': sorted(md['artist_name'].fillna('unknown').unique())}).set_index('artist_name')
tbl['days_at_55'] = tbl.index.to_series().map(days_55).fillna(0).astype(int)
tbl['days_at_75'] = tbl.index.to_series().map(days_75).fillna(0).astype(int)

# ---- Budget inputs w/ HUMAN-REVIEW safe defaults --------------------------
TOTAL_BUDGET   = 100_000.0
TIER55_PCT     = 0.45
TIER75_PCT     = 0.55
CAP_PER_ARTIST = None

BUDGET_OK = False
if "BUDGET" in globals():
    TOTAL_BUDGET   = float(BUDGET.get("total_pool") or TOTAL_BUDGET)
    TIER55_PCT     = float(BUDGET.get("tier_55_pct") or TIER55_PCT)
    TIER75_PCT     = float(BUDGET.get("tier_75_pct") or TIER75_PCT)
    CAP_PER_ARTIST = BUDGET.get("cap_per_artist", CAP_PER_ARTIST)
if "HUMAN_REVIEW_APPROVED" in globals():
    BUDGET_OK = bool(HUMAN_REVIEW_APPROVED.get("budget_inputs", False))

TIER55_POOL = TOTAL_BUDGET * TIER55_PCT
TIER75_POOL = TOTAL_BUDGET * TIER75_PCT

# ---- Strategy A: Tiered (55 test, 75 scale) -------------------------------
a = tbl.copy()
a["only_55_days"] = (a["days_at_55"] - a["days_at_75"]).clip(lower=0)

den_55 = a["only_55_days"].sum()
den_75 = a["days_at_75"].sum()
a["alloc_55"] = 0.0 if den_55==0 else (a["only_55_days"]/den_55) * TIER55_POOL
a["alloc_75"] = 0.0 if den_75==0 else (a["days_at_75"]/den_75) * TIER75_POOL
a["total_allocation"] = a["alloc_55"] + a["alloc_75"]

if CAP_PER_ARTIST is not None:
    a["total_allocation"] = a["total_allocation"].clip(upper=float(CAP_PER_ARTIST))
strategy_a = a[["total_allocation"]].reset_index().sort_values("total_allocation", ascending=True)

# ---- Strategy B: Consistency Bonus (55 & 75 frequency mix) ----------------
b = tbl.copy()
max55 = max(b["days_at_55"].max(), 1)
max75 = max(b["days_at_75"].max(), 1)
b["consistency_score"] = 0.6*(b["days_at_55"]/max55) + 0.4*(b["days_at_75"]/max75)
total_score = max(b["consistency_score"].sum(), 1e-9)
b["allocation"] = (b["consistency_score"]/total_score) * TOTAL_BUDGET
if CAP_PER_ARTIST is not None:
    b["allocation"] = b["allocation"].clip(upper=float(CAP_PER_ARTIST))
strategy_b = b[["allocation"]].reset_index().sort_values("allocation", ascending=True)

# ---- Strategy C: Consistency Ranking (proportional to 55-days) ------------
c = tbl.copy()
den = max(c["days_at_55"].sum(), 1)
c["allocation"] = (c["days_at_55"]/den) * TOTAL_BUDGET
if CAP_PER_ARTIST is not None:
    c["allocation"] = c["allocation"].clip(upper=float(CAP_PER_ARTIST))
strategy_c = c[["allocation"]].reset_index().sort_values("allocation", ascending=True)

# ---- Plot: three small-multiple barh with direct labels -------------------
BLUE  = "#1f77b4"; ORNG = "#ff7f0e"; PURP = "#7b3294"; TXT="#222"; GRID="#e7e7e7"

def _barh(ax, df, val_col, color, title):
    y = np.arange(len(df))
    ax.barh(y, df[val_col].values, color=color, alpha=0.9, height=0.6)
    ax.set_yticks(y); ax.set_yticklabels(df["artist_name"], fontsize=10)
    ax.grid(True, axis="x", alpha=0.3, color=GRID)
    ax.set_xlabel("Allocation ($)", fontsize=11, color=TXT)
    ax.set_title(title, fontweight="bold")
    # direct labels
    for yy, v in zip(y, df[val_col].values):
        ax.text(v, yy, f" ${v:,.0f}", ha="left", va="center", fontsize=10, color=TXT)

fig, axes = plt.subplots(1,3, figsize=(18,7), sharex=False)
_barh(axes[0], strategy_a, "total_allocation", BLUE,  "A) Tiered Escalation → test at 55, scale at 75")
_barh(axes[1], strategy_b, "allocation",      ORNG,  "B) Consistency Bonus → reward reliability")
_barh(axes[2], strategy_c, "allocation",      PURP,  "C) Consistency Ranking → simple & transparent")
plt.tight_layout(); plt.show()

print("✅ Strategy comparison rendered (A/B/C).")
if not BUDGET_OK:
    print("⚠️  Using safe defaults — set BUDGET and approve HUMAN_REVIEW to finalize.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# S2 — Pros/Cons + Decision Cues (cards + quick checklist)
# ═══════════════════════════════════════════════════════════════════════════
from IPython.display import HTML, display

BLUE  = "#1f77b4"; ORNG = "#ff7f0e"; PURP = "#7b3294"; TXT="#111"

html = f"""
<style>
.strategy-grid, .strategy-grid * {{
    color: #000000 !important;
}}
</style>
<div class="strategy-grid" style="font-family: ui-sans-serif, system-ui; color:#000000 !important;">
  <h3 style="margin:0 0 10px 0; color:#000000 !important;">📊 Strategy Pros & Cons — pick one to test first</h3>
  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:14px;">
    <div style="border:2px solid {BLUE};border-radius:14px;padding:14px;background:#fff;">
      <h4 style="margin:0 0 6px 0;color:{BLUE} !important;">A) Tiered Escalation</h4>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">✅ Pros:</b> managed risk, gates at 55/75, rewards sustained heat</p>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">❌ Cons:</b> requires monitoring, may miss ultra-fast spikes</p>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">Use when:</b> you can watch daily & want disciplined scale-ups</p>
    </div>
    <div style="border:2px solid {ORNG};border-radius:14px;padding:14px;background:#fff;">
      <h4 style="margin:0 0 6px 0;color:{ORNG} !important;">B) Consistency Bonus</h4>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">✅ Pros:</b> favors reliability, balances 55 & 75 history</p>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">❌ Cons:</b> slower to capitalize on viral moments</p>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">Use when:</b> you want steady growth & fewer one-hit bets</p>
    </div>
    <div style="border:2px solid {PURP};border-radius:14px;padding:14px;background:#fff;">
      <h4 style="margin:0 0 6px 0;color:{PURP} !important;">C) Consistency Ranking</h4>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">✅ Pros:</b> simplest to explain & run</p>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">❌ Cons:</b> can underweight rising artists</p>
      <p style="color:#000000 !important;"><b style="color:#000000 !important;">Use when:</b> you need quick clarity & low ops overhead</p>
    </div>
  </div>
  <div style="margin-top:12px;padding:12px;border:1px dashed #999;border-radius:12px;background:#fafafa;">
    <b style="color:#000000 !important;">Decision checklist (1 minute):</b>
    <ol style="margin:6px 0 0 18px; color:#000000 !important;">
      <li style="color:#000000 !important;">Ops capacity this month: <i style="color:#000000 !important;">monitor daily or weekly?</i></li>
      <li style="color:#000000 !important;">Risk appetite: <i style="color:#000000 !important;">spread steady vs. chase spikes?</i></li>
      <li style="color:#000000 !important;">Team goal: <i style="color:#000000 !important;">talent development vs. fast discovery?</i></li>
    </ol>
    <div style="margin-top:6px;color:#000000 !important;">
      Tip: Use <b style="color:#000000 !important;">annotations/direct labels</b> in the next charts so the story reads fast — titles should say
      the finding, why it matters, and what to do next.
    </div>
  </div>
</div>
"""
display(HTML(html))
print("✅ Strategy pros/cons summary displayed")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# STRATEGY PROS/CONS SUMMARY (with black text fix)
# ═══════════════════════════════════════════════════════════════════════════

from IPython.display import HTML, display

# Define PALETTE if not already defined
if 'PALETTE' not in globals():
    PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c']

html = f"""
<style>
.strategy-summary, .strategy-summary * {{
    color: #000000 !important;
}}
</style>
<div class="strategy-summary" style="padding:20px;border:2px solid {PALETTE[0]};border-radius:12px;background:#f9f9f9;color:#000000 !important;">
    <h3 style="color:#000000 !important;">📊 Strategy Comparison Summary</h3>
    
    <h4 style="color:{PALETTE[0]} !important;">Strategy A: Tiered Escalation</h4>
    <p style="color:#000000 !important;"><b style="color:#000000 !important;">✅ Pros:</b></p>
    <ul style="color:#000000 !important;">
        <li style="color:#000000 !important;">Gradual risk management (test at 55, scale at 75)</li>
        <li style="color:#000000 !important;">Rewards sustained performance over time</li>
        <li style="color:#000000 !important;">Clear decision gates for budget increases</li>
    </ul>
    <p style="color:#000000 !important;"><b style="color:#000000 !important;">❌ Cons:</b></p>
    <ul style="color:#000000 !important;">
        <li style="color:#000000 !important;">Complex tracking across two tiers</li>
        <li style="color:#000000 !important;">May miss fast-moving viral opportunities</li>
        <li style="color:#000000 !important;">Requires real-time monitoring infrastructure</li>
    </ul>
    
    <h4 style="color:{PALETTE[1]} !important;">Strategy B: Consistency Bonus</h4>
    <p style="color:#000000 !important;"><b style="color:#000000 !important;">✅ Pros:</b></p>
    <ul style="color:#000000 !important;">
        <li style="color:#000000 !important;">Rewards reliable performers (reduces one-hit-wonder risk)</li>
        <li style="color:#000000 !important;">Balances frequency (55) with quality (75)</li>
        <li style="color:#000000 !important;">Encourages sustainable artist development</li>
    </ul>
    <p style="color:#000000 !important;"><b style="color:#000000 !important;">❌ Cons:</b></p>
    <ul style="color:#000000 !important;">
        <li style="color:#000000 !important;">May penalize emerging artists with limited history</li>
        <li style="color:#000000 !important;">Slower to capitalize on viral moments</li>
        <li style="color:#000000 !important;">Requires longer evaluation period</li>
    </ul>
    
    <h4 style="color:{PALETTE[2]} !important;">Strategy C: Consistency-Based Allocation</h4>
    <p style="color:#000000 !important;"><b style="color:#000000 !important;">✅ Pros:</b></p>
    <ul style="color:#000000 !important;">
        <li style="color:#000000 !important;">Data-driven, transparent ranking system</li>
        <li style="color:#000000 !important;">Simple to explain and implement</li>
        <li style="color:#000000 !important;">Focuses on pre-breakout signal (55 threshold)</li>
    </ul>
    <p style="color:#000000 !important;"><b style="color:#000000 !important;">❌ Cons:</b></p>
    <ul style="color:#000000 !important;">
        <li style="color:#000000 !important;">May create internal competition between artists</li>
        <li style="color:#000000 !important;">Doesn't account for growth potential or trajectory</li>
        <li style="color:#000000 !important;">Winner-take-all dynamics could demotivate lower-ranked artists</li>
    </ul>
</div>
"""

display(HTML(html))
print("✅ Strategy pros/cons summary displayed")

---

# 🎉 Fun FYI: Additional Insights

Celebratory, non-blocking insights for deeper exploration.

---

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 24 — FYI 1: Comment length distribution (zoomed to 800 chars, outlier count)
# ═══════════════════════════════════════════════════════════════════════════
if comments_safe is None or "text" not in comments_safe.columns:
    print("ℹ️ FYI 1 skipped: comments not available.")
else:
    DISPLAY_LIMIT = 800

    lens = comments_safe["text"].astype(str).str.len()
    total_comments = len(lens)
    outliers = int((lens > DISPLAY_LIMIT).sum())
    outlier_pct = (outliers / total_comments * 100) if total_comments > 0 else 0.0

    lens_display = lens.clip(upper=DISPLAY_LIMIT)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(lens_display, bins=40, color=GREY, edgecolor=TXT, alpha=0.85, range=(0, DISPLAY_LIMIT))
    ax.set_xlabel("Comment length (characters)", fontsize=11)
    ax.set_ylabel("Count", fontsize=11)

    outlier_note = (
        f" ({outliers:,} comments >{DISPLAY_LIMIT} chars = {outlier_pct:.1f}%)"
        if outliers > 0 else ""
    )
    clean_title(
        ax,
        f"Most comments are short (0-{DISPLAY_LIMIT} chars){outlier_note} → sample long-form feedback for qualitative insight",
    )

    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.show()
    print(
        f"✅ FYI 1: Comment length (zoomed to {DISPLAY_LIMIT} chars, {outliers:,} outliers)"
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 25 — FYI 2: Publish hour vs performance (direct-labeled; palette-consistent)
# ═══════════════════════════════════════════════════════════════════════════
df = vids_safe.copy()
if "views_per_day" not in df.columns:
    now = pd.Timestamp.today().normalize()
    df["age_days"] = (now - df["pub_date"]).dt.days.clip(lower=1)
    df["views_per_day"] = (df["view_count"]/df["age_days"]).replace([np.inf,np.nan],0.0)

df["hour"] = df["published_at"].dt.hour
hour_perf = df.groupby("hour")["views_per_day"].mean().reindex(range(24), fill_value=0.0)

fig, ax = plt.subplots(figsize=(12,5))
bars = ax.bar(hour_perf.index, hour_perf.values, color=GREY, edgecolor=TXT, alpha=0.7)
best_h = int(hour_perf.idxmax()); best_v = float(hour_perf.max())
bars[best_h].set_color(BLUE)

for h, v in hour_perf.items():
    ax.text(h, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9, color=TXT)

ax.set_xlabel("Publish Hour (24h)"); ax.set_ylabel("Avg Views/Day")
clean_title(ax, f"Hour {best_h:02d}:00 has the highest views/day → schedule priority releases around {best_h:02d}:00")
plt.tight_layout(); plt.show()

print("✅ FYI 2: Publish hour vs performance (direct-labeled)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# NOTEBOOK HYGIENE REMINDER
# ═══════════════════════════════════════════════════════════════════════════

print("""
✅ Notebook hygiene (run once per repo):

1) Strip outputs & pair to text for clean diffs:
   pip install nbstripout jupytext
   nbstripout --install
   jupytext --set-formats 'ipynb,py:percent' MusicScope_YouTube_Dashboard.ipynb

2) Lint / type-check notebooks via nbQA:
   pip install nbqa ruff mypy
   nbqa ruff MusicScope_YouTube_Dashboard.ipynb --fix
   nbqa mypy MusicScope_YouTube_Dashboard.ipynb --ignore-missing-imports

3) Commit with conventional commit message:
   git add MusicScope_YouTube_Dashboard.ipynb
   git commit -m "feat(dashboard): add MusicScope YouTube analytics dashboard"
""")